Aim of this script: attribute an RFN line number (`code_ligne`) to each station in the
`data_chuuchuu_{data_selection}_enriched.parquet` dataset, by spatially matching each
station's coordinates against the track geometry in `geo_data/rfn_caracteristiques.gpkg`.

This is a first pass only, at the **station** level (not per train-leg / per-segment --
see the earlier discussion notebook on why that's a separate, harder problem). Three
outcomes per station:

- **no match at all** within the buffer distance -> we simply won't have infrastructure
  data for that station's rows. That's fine, left as null.
- **exactly one line matches** -> unambiguous, assign it.
- **two or more lines match** (junction stations) -> label the row `ambiguous` rather
  than guessing. Deciding between candidates is deferred to a later notebook.

In [183]:
import os
import pandas as pd
import geopandas as gpd


## Step 0 -- Load the enriched Chuuchuu dataset

In [184]:
data_selection = "french"

intermediate_outputs_dir = "intermediate_outputs"
data_path = f"{intermediate_outputs_dir}/data_chuuchuu_{data_selection}_enriched.parquet"

try:
    data_chuuchuu.head()
except NameError:
    data_chuuchuu = pd.read_parquet(data_path)
data_chuuchuu.shape


(7038209, 50)

## Step 1 -- Station coordinates & RFN line geometry

The first matching is done once per **unique station**, not once per row yet: there are ~3,800
unique stations against 7M+ stop-event rows, so we build the station-level match table
first and broadcast it back onto the full dataframe at the end.

`stations.csv`'s `db_id` column is the same identifier as `deutscheBahnStopId` in the
Chuuchuu data (confirmed earlier by spot-checking coordinates against known stations,
e.g. Paris Montparnasse).

In [185]:
unique_stations = data_chuuchuu.drop_duplicates(subset=["deutscheBahnStopId"])[
    ["deutscheBahnStopId", "stopName", "country"]
].copy()
unique_stations["db_id_str"] = unique_stations["deutscheBahnStopId"].astype(str)
print(f"{len(unique_stations)} unique stations across all countries in the dataset")


3791 unique stations across all countries in the dataset


In [186]:
data_stations = pd.read_csv("sup_data/stations.csv", sep=";", low_memory=False)
data_stations = data_stations.dropna(subset=["db_id", "latitude", "longitude"])
data_stations["db_id_str"] = data_stations["db_id"].astype("int64").astype(str)
data_stations_small = data_stations[["db_id_str", "latitude", "longitude"]].drop_duplicates(subset=["db_id_str"])

stations_with_coords = unique_stations.merge(data_stations_small, on="db_id_str", how="left")
has_coords_mask = stations_with_coords["latitude"].notna() & stations_with_coords["longitude"].notna()
print(f"{has_coords_mask.sum()} / {len(stations_with_coords)} unique stations resolved to coordinates via stations.csv")


3186 / 3791 unique stations resolved to coordinates via stations.csv


In [187]:
rfn_lines = gpd.read_file("geo_data/rfn_caracteristiques.gpkg")
rfn_lines_proj = rfn_lines.to_crs(2154)  # Lambert-93, metric CRS for a metre-based buffer
print(f"{len(rfn_lines)} track segments across {rfn_lines['code_ligne'].nunique()} lines")


1442 track segments across 639 lines


## Step 2 -- Spatial match: station -> candidate line(s)

`MATCH_BUFFER_M` is the distance (in metres) a station must fall within of a line's
track geometry to count as "on" that line. 150 m was validated earlier (checked against
50/100/300 m -- 150 m is tight enough to avoid picking up unrelated parallel lines while
still catching genuine trackside station positions).

Note this uses `sjoin` with an actual buffer polygon + `intersects`, not
`sjoin_nearest` -- `sjoin_nearest` only ever returns the single closest line, which
would silently hide every ambiguous/junction case.

In [188]:
MATCH_BUFFER_M = 150

stations_geo = stations_with_coords.loc[has_coords_mask].reset_index(drop=True)
stations_gdf = gpd.GeoDataFrame(
    stations_geo,
    geometry=gpd.points_from_xy(stations_geo["longitude"], stations_geo["latitude"]),
    crs="EPSG:4326",
).to_crs(2154)

stations_buffered = stations_gdf.copy()
stations_buffered["geometry"] = stations_buffered.geometry.buffer(MATCH_BUFFER_M)

matches = gpd.sjoin(
    stations_buffered[["db_id_str", "geometry"]],
    rfn_lines_proj[["code_ligne", "geometry"]],
    how="left",
    predicate="intersects",
)
matches = matches.drop(columns=["index_right"])
print(f"{len(matches)} (station, candidate line) pairs from {len(stations_buffered)} stations")


3494 (station, candidate line) pairs from 3186 stations


In [189]:
candidates_per_station = (
    matches.groupby("db_id_str")["code_ligne"]
    .apply(lambda s: sorted(s.dropna().unique()))
    .rename("code_ligne_candidates")
)

station_match = stations_geo[["db_id_str", "deutscheBahnStopId", "stopName", "country"]].merge(
    candidates_per_station, on="db_id_str", how="left"
)
station_match["code_ligne_candidates"] = station_match["code_ligne_candidates"].apply(
    lambda v: v if isinstance(v, list) else []
)
station_match["n_candidates"] = station_match["code_ligne_candidates"].apply(len)
station_match.head()


,db_id_str,deutscheBahnStopId,stopName,country,code_ligne_candidates,n_candidates
0,8700045,8700045,Ambérieu-en-Bugey,France,"[883000, 889000, 890000]",3
1,8702414,8702414,Pont-d'Ain,France,[883000],1
2,8702407,8702407,La Vavrette Tossiat RD1075,France,[883000],1
3,8769599,8769599,Bourg-en-Bresse Archives,France,[],0
4,8700037,8700037,Bourg-en-Bresse Gare Routière,France,"[880000, 883000]",2


## Step 3 -- Label each station: `no_coordinates` / `no_match` / `unique` / `ambiguous`

- `no_coordinates`: never resolved to a lat/lon via `stations.csv` in the first place
  (e.g. foreign stations not covered by the sup_data file, or a stale/renamed
  `deutscheBahnStopId`).
- `no_match`: coordinates exist but sit further than `MATCH_BUFFER_M` from every line
  (expected for non-French stations -- the gpkg only covers the French network -- and
  for tram/metro stops outside RFN scope).
- `unique`: exactly one candidate line -> `code_ligne` is populated.
- `ambiguous`: 2+ candidate lines -> `code_ligne` stays null, candidates kept in
  `code_ligne_candidates` for the later disambiguation step.

In [190]:
no_coords_stations = stations_with_coords.loc[~has_coords_mask, ["db_id_str", "deutscheBahnStopId", "stopName", "country"]].copy()
no_coords_stations["code_ligne_candidates"] = [[] for _ in range(len(no_coords_stations))]
no_coords_stations["n_candidates"] = 0
no_coords_stations["line_match_status"] = "no_coordinates"

def status_from_n(n):
    if n == 0:
        return "no_match"
    if n == 1:
        return "unique"
    return "ambiguous"

station_match["line_match_status"] = station_match["n_candidates"].apply(status_from_n)
station_match["code_ligne"] = station_match.apply(
    lambda r: r["code_ligne_candidates"][0] if r["line_match_status"] == "unique" else pd.NA, axis=1
)

station_line_lookup = pd.concat(
    [
        station_match[["deutscheBahnStopId", "stopName", "country", "code_ligne", "code_ligne_candidates", "n_candidates", "line_match_status"]],
        no_coords_stations.assign(code_ligne=pd.NA)[["deutscheBahnStopId", "stopName", "country", "code_ligne", "code_ligne_candidates", "n_candidates", "line_match_status"]],
    ],
    ignore_index=True,
)
assert len(station_line_lookup) == len(unique_stations)
station_line_lookup["line_match_status"].value_counts(dropna=False)


line_match_status
unique            2325
no_match           689
no_coordinates     605
ambiguous          172
Name: count, dtype: int64

### Verification -- proportions and a look at the ambiguous cases

In [191]:
counts = station_line_lookup["line_match_status"].value_counts()
pct = station_line_lookup["line_match_status"].value_counts(normalize=True) * 100
summary_table = pd.DataFrame({"n_stations": counts, "pct": pct.round(1)})
summary_table


,n_stations,pct
line_match_status,,
unique,2325,61.3
no_match,689,18.2
no_coordinates,605,16.0
ambiguous,172,4.5


In [192]:
# France-only view, since the gpkg only covers the French network -- the "no_match"
# bucket above is dominated by stations in other countries, which is expected and not
# a data quality issue
france_lookup = station_line_lookup[station_line_lookup["country"] == "France"]
print(f"France-only: {len(france_lookup)} stations")
france_lookup["line_match_status"].value_counts(normalize=True).mul(100).round(1)


France-only: 3583 stations


line_match_status
unique            64.9
no_coordinates    15.7
no_match          14.6
ambiguous          4.8
Name: proportion, dtype: float64

In [193]:
station_line_lookup[station_line_lookup["line_match_status"] == "ambiguous"].sort_values(
    "n_candidates", ascending=False
).head(20)


,deutscheBahnStopId,stopName,country,code_ligne,code_ligne_candidates,n_candidates,line_match_status
1331,8700015,Paris Saint-Lazare,France,NaN,"[334000, 334900, 340000, 973000, 975000]",5,ambiguous
0,8700045,Ambérieu-en-Bugey,France,NaN,"[883000, 889000, 890000]",3,ambiguous
528,8700023,Strasbourg,France,NaN,"[070000, 142000, 145000]",3,ambiguous
415,8700469,Is-sur-Tille,France,NaN,"[838000, 843000, 849000]",3,ambiguous
770,8700125,Somain,France,NaN,"[250000, 258000, 262000]",3,ambiguous
1086,8700047,Bordeaux Saint-Jean,France,NaN,"[570000, 640000, 655000]",3,ambiguous
1143,8700058,Tarascon-sur-Rhône,France,NaN,"[810000, 830351, 830356]",3,ambiguous
1121,8700072,Creil,France,NaN,"[272000, 316000, 329000]",3,ambiguous
749,8700439,Sarreguemines,France,NaN,"[159000, 161000, 163000]",3,ambiguous
839,8703900,Mantes-la-Jolie,France,NaN,"[334000, 340000, 366000]",3,ambiguous


## Step 4 -- Broadcast the station-level match back onto every stop-event row

One row per (train, stop) in `data_chuuchuu`, so the same station's line match applies
to every train that ever stopped there.

In [194]:
rows_before = len(data_chuuchuu)

# drop first so this cell is safe to re-run without restarting the kernel -- otherwise
# re-running after Step 5 (or after a previous run of this same cell) collides with the
# columns this merge is about to (re-)add
data_chuuchuu = data_chuuchuu.drop(
    columns=["code_ligne", "code_ligne_candidates", "n_candidates", "line_match_status"], errors="ignore"
)

data_chuuchuu = data_chuuchuu.merge(
    station_line_lookup[["deutscheBahnStopId", "code_ligne", "code_ligne_candidates", "n_candidates", "line_match_status"]],
    on="deutscheBahnStopId",
    how="left",
)

assert len(data_chuuchuu) == rows_before, "merge changed row count -- deutscheBahnStopId isn't unique in the lookup"
data_chuuchuu["line_match_status"].value_counts(dropna=False)


line_match_status
unique            5184359
ambiguous         1013521
no_match           509134
no_coordinates     331195
Name: count, dtype: int64

In [195]:
data_chuuchuu.loc[data_chuuchuu["country"] == "France", "line_match_status"].value_counts(normalize=True).mul(100).round(1)


line_match_status
unique            79.5
ambiguous         15.5
no_coordinates     3.0
no_match           1.9
Name: proportion, dtype: float64

## Step 4.5 -- Reclassify non-French `no_match`/`no_coordinates` stops as `international`

The RFN geometry (`rfn_caracteristiques.gpkg`) only covers the French network, so a
non-French station showing `no_match` or `no_coordinates` isn't a resolution failure --
there was never an RFN line for it to match in the first place (see Step 3's note on
this). Relabelling these upfront, before 5a, means 5a-5g's propagation logic never
wastes a pass trying to resolve a station that can, by definition, never match an RFN
line -- and downstream readers see an explicit `international` label instead of a
`no_match`/`no_coordinates` that looks like a data quality gap.

**Design note -- `code_ligne` is deliberately left null here, not set to a literal
string yet.** 5a's "both neighbours agree" rule (and the equivalent no_match/no_coordinates
step further down) reads a *neighbour's* `code_ligne` with only a `.notna()` check --
there's no candidate list to cross-check against for `no_match`/`no_coordinates` rows,
unlike 5b/5c. If `code_ligne` were set to a literal `"international line"` string right
now, a French stop sitting next to a reclassified station could silently inherit that
string through neighbour-propagation as if it were a real RFN code. Leaving `code_ligne`
null keeps every downstream propagation rule working exactly as it already does for
`no_match`/`no_coordinates` rows -- `line_match_status = "international"` on its own is
enough to exclude these rows from every 5a-5g mask, since none of those check for it.
The `"international line"` label is stamped into `code_ligne` as a purely cosmetic step
at the very end of the notebook, once no further propagation can happen.

In [196]:
international_mask = (
    (data_chuuchuu["country"] != "France")
    & data_chuuchuu["line_match_status"].isin(["no_match", "no_coordinates"])
)
data_chuuchuu.loc[international_mask, "line_match_status"] = "international"

print(f"{international_mask.sum()} rows reclassified as international (non-French, no RFN line to match)")
data_chuuchuu["line_match_status"].value_counts(dropna=False)

520363 rows reclassified as international (non-French, no RFN line to match)


line_match_status
unique            5184359
ambiguous         1013521
international      520363
no_coordinates     194464
no_match           125502
Name: count, dtype: int64

## Step 5 -- for trains passing through stations with several potential line matches (<i>Ambiguous</i>) : identify which lines the train took 

### 5a -- Resolve using matching neighbours

If an `ambiguous` or `no_match` stop is an intermediate stop on its journey (not the
depart or terminus), and the stop immediately before it and the stop immediately after
it on the *same* journey were both resolved to the same `code_ligne`, assign that line
to this stop too -- a train can't leave one line and come back to it within a single
intermediate stop.

In [197]:
data_chuuchuu = data_chuuchuu.sort_values(["journey_id", "sort_time"]).reset_index(drop=True)
journey_code_ligne = data_chuuchuu.groupby("journey_id")["code_ligne"]

prev_code_ligne = journey_code_ligne.shift(1)
next_code_ligne = journey_code_ligne.shift(-1)

resolved_by_neighbours_mask = (
    data_chuuchuu["line_match_status"].isin(["ambiguous", "no_match"])
    & (data_chuuchuu["depart_terminus"] == "intermediate")
    & prev_code_ligne.notna()
    & next_code_ligne.notna()
    & (prev_code_ligne == next_code_ligne)
)

data_chuuchuu.loc[resolved_by_neighbours_mask, "code_ligne"] = prev_code_ligne[resolved_by_neighbours_mask]
data_chuuchuu.loc[resolved_by_neighbours_mask, "line_match_status"] = (
    data_chuuchuu.loc[resolved_by_neighbours_mask, "line_match_status"] + "_resolved_by_neighbours"
)

print(f"{resolved_by_neighbours_mask.sum()} rows resolved: same line on the stop directly before and after")
data_chuuchuu["line_match_status"].value_counts(dropna=False)


145362 rows resolved: same line on the stop directly before and after


line_match_status
unique                              5184359
ambiguous                            874068
international                        520363
no_coordinates                       194464
ambiguous_resolved_by_neighbours     139453
no_match                             119593
no_match_resolved_by_neighbours        5909
Name: count, dtype: int64

### 5b -- Resolve ambiguous depart/terminus stops when an endpoint neighbour's line is one of the candidates

A `depart` stop only has a *next* stop to compare against; a `terminus` stop only has a
*previous* one. So instead of 5a's "same line before and after" rule, check the single
available neighbour: if its `code_ligne` is itself one of this stop's own
`code_ligne_candidates`, the train was almost certainly already on that line at
departure (or still on it at the terminus) -- assign it.

This is weaker evidence than 5a (only one neighbour, not two agreeing) and weaker than
5c's topological check (no geometric proof of a connection), but it's cheap, safe, and
catches a lot of cases before falling back to 5c's more expensive check -- e.g. a train
arriving at a junction terminus straight off a line that's already one of the
terminus's own candidates.

In [198]:
journey_code_ligne = data_chuuchuu.groupby("journey_id")["code_ligne"]
prev_code_ligne = journey_code_ligne.shift(1)
next_code_ligne = journey_code_ligne.shift(-1)

depart_eligible = (
    (data_chuuchuu["line_match_status"] == "ambiguous")
    & (data_chuuchuu["depart_terminus"] == "depart")
    & next_code_ligne.notna()
)
terminus_eligible = (
    (data_chuuchuu["line_match_status"] == "ambiguous")
    & (data_chuuchuu["depart_terminus"] == "terminus")
    & prev_code_ligne.notna()
)

resolved_depart_mask = depart_eligible.copy()
resolved_depart_mask.loc[depart_eligible] = [
    next_line in candidates
    for next_line, candidates in zip(
        next_code_ligne[depart_eligible], data_chuuchuu.loc[depart_eligible, "code_ligne_candidates"]
    )
]

resolved_terminus_mask = terminus_eligible.copy()
resolved_terminus_mask.loc[terminus_eligible] = [
    prev_line in candidates
    for prev_line, candidates in zip(
        prev_code_ligne[terminus_eligible], data_chuuchuu.loc[terminus_eligible, "code_ligne_candidates"]
    )
]

data_chuuchuu.loc[resolved_depart_mask, "code_ligne"] = next_code_ligne[resolved_depart_mask]
data_chuuchuu.loc[resolved_terminus_mask, "code_ligne"] = prev_code_ligne[resolved_terminus_mask]

resolved_endpoint_mask = resolved_depart_mask | resolved_terminus_mask
data_chuuchuu.loc[resolved_endpoint_mask, "line_match_status"] = (
    data_chuuchuu.loc[resolved_endpoint_mask, "line_match_status"] + "_resolved_by_endpoint_candidate"
)

print(f"{resolved_depart_mask.sum()} depart rows resolved: next stop's line is one of this stop's candidates")
print(f"{resolved_terminus_mask.sum()} terminus rows resolved: previous stop's line is one of this stop's candidates")
data_chuuchuu["line_match_status"].value_counts(dropna=False)


111224 depart rows resolved: next stop's line is one of this stop's candidates
110457 terminus rows resolved: previous stop's line is one of this stop's candidates


line_match_status
unique                                      5184359
ambiguous                                    652387
international                                520363
ambiguous_resolved_by_endpoint_candidate     221681
no_coordinates                               194464
ambiguous_resolved_by_neighbours             139453
no_match                                     119593
no_match_resolved_by_neighbours                5909
Name: count, dtype: int64

### 5c -- Resolve ambiguous intermediate stops via chain consistency (partial neighbour evidence)

Spotted a case neither 5a nor 5b catches: two ambiguous intermediate stops in a row --
Avignon Centre (`[824306, 830000]`) then Sorgues (`[830000, 927000]`) -- followed by a
resolved one, Courthézon (`830000`). 5a needs *both* neighbours resolved and agreeing;
Sorgues only has one (Courthézon). 5b only covers depart/terminus stops.

There's a middle-ground signal available though: Sorgues has one neighbour resolved
(Courthézon = `830000`) and one still ambiguous (Avignon). `830000` being one of
Sorgues' own candidates is suggestive on its own, but to guard against picking the
wrong candidate at a real junction, this rule adds a corroborating check: the *same*
line must also appear in the still-ambiguous neighbour's own candidate list -- Avignon's
candidates include `830000` too. Three-way agreement (this stop's own candidates, the
resolved neighbour's line, and the other, still-ambiguous neighbour's candidates) is
stronger evidence than a plain two-way match, at the cost of only firing when that third
piece of corroboration happens to be available.

In [199]:
journey_code_ligne = data_chuuchuu.groupby("journey_id")["code_ligne"]
journey_status = data_chuuchuu.groupby("journey_id")["line_match_status"]
journey_candidates = data_chuuchuu.groupby("journey_id")["code_ligne_candidates"]

prev_code_ligne = journey_code_ligne.shift(1)
next_code_ligne = journey_code_ligne.shift(-1)
prev_status = journey_status.shift(1)
next_status = journey_status.shift(-1)
prev_candidates = journey_candidates.shift(1)
next_candidates = journey_candidates.shift(-1)

base_eligible = (data_chuuchuu["line_match_status"] == "ambiguous") & (data_chuuchuu["depart_terminus"] == "intermediate")
forward_eligible = base_eligible & next_code_ligne.notna() & (prev_status == "ambiguous")
backward_eligible = base_eligible & prev_code_ligne.notna() & (next_status == "ambiguous")

def check_forward(i):
    anchor = next_code_ligne.iloc[i]
    other_candidates = prev_candidates.iloc[i]
    return anchor in data_chuuchuu["code_ligne_candidates"].iloc[i] and isinstance(other_candidates, list) and anchor in other_candidates

def check_backward(i):
    anchor = prev_code_ligne.iloc[i]
    other_candidates = next_candidates.iloc[i]
    return anchor in data_chuuchuu["code_ligne_candidates"].iloc[i] and isinstance(other_candidates, list) and anchor in other_candidates

resolved_forward_mask = forward_eligible.copy()
resolved_forward_mask.loc[forward_eligible] = [check_forward(i) for i in data_chuuchuu.index[forward_eligible]]

resolved_backward_mask = backward_eligible.copy()
resolved_backward_mask.loc[backward_eligible] = [check_backward(i) for i in data_chuuchuu.index[backward_eligible]]

data_chuuchuu.loc[resolved_forward_mask, "code_ligne"] = next_code_ligne[resolved_forward_mask]
data_chuuchuu.loc[resolved_backward_mask, "code_ligne"] = prev_code_ligne[resolved_backward_mask]

resolved_chain_mask = resolved_forward_mask | resolved_backward_mask
data_chuuchuu.loc[resolved_chain_mask, "line_match_status"] = (
    data_chuuchuu.loc[resolved_chain_mask, "line_match_status"] + "_resolved_by_chain_consistency"
)

print(f"{resolved_forward_mask.sum()} rows resolved forward (next resolved, prev ambiguous, 3-way match)")
print(f"{resolved_backward_mask.sum()} rows resolved backward (prev resolved, next ambiguous, 3-way match)")
data_chuuchuu["line_match_status"].value_counts(dropna=False)


33494 rows resolved forward (next resolved, prev ambiguous, 3-way match)
33303 rows resolved backward (prev resolved, next ambiguous, 3-way match)


line_match_status
unique                                      5184359
ambiguous                                    585590
international                                520363
ambiguous_resolved_by_endpoint_candidate     221681
no_coordinates                               194464
ambiguous_resolved_by_neighbours             139453
no_match                                     119593
ambiguous_resolved_by_chain_consistency       66797
no_match_resolved_by_neighbours                5909
Name: count, dtype: int64

### 5d -- Loop 5a, 5b and 5c together until nothing new resolves

Resolving one ambiguous stop can turn it into a usable, resolved neighbour for the
*next* stop over. In the Avignon/Sorgues example above, once 5c resolves Sorgues to
`830000`, Avignon Centre (a `depart` stop) now has a resolved next-stop whose line
(`830000`) is one of its own candidates -- exactly 5b's rule, just not available until
5c ran. So 5a, 5b and 5c are re-run together, repeatedly, recomputing neighbours from
the latest `code_ligne` each pass, until a full pass resolves nothing new -- this
chains resolutions along a corridor instead of only ever getting the first link.

5a and 5b already ran once above (their standalone cells show the first-pass numbers on
their own); this cell repeats all three rules from the current state, so its first
printed pass finds only what's left over, and subsequent passes pick up the knock-on
effects.

In [200]:
MAX_CHEAP_ITERATIONS = 15  # safety net -- converges in a handful of passes in practice

def resolve_neighbours_pass():
    """5a: both neighbours resolved and agree."""
    journey_code_ligne = data_chuuchuu.groupby("journey_id")["code_ligne"]
    prev_code_ligne = journey_code_ligne.shift(1)
    next_code_ligne = journey_code_ligne.shift(-1)
    mask = (
        data_chuuchuu["line_match_status"].isin(["ambiguous", "no_match"])
        & (data_chuuchuu["depart_terminus"] == "intermediate")
        & prev_code_ligne.notna() & next_code_ligne.notna()
        & (prev_code_ligne == next_code_ligne)
    )
    data_chuuchuu.loc[mask, "code_ligne"] = prev_code_ligne[mask]
    data_chuuchuu.loc[mask, "line_match_status"] = data_chuuchuu.loc[mask, "line_match_status"] + "_resolved_by_neighbours"
    return int(mask.sum())

def resolve_endpoint_pass():
    """5b: depart/terminus stop whose single neighbour's line is one of its own candidates."""
    journey_code_ligne = data_chuuchuu.groupby("journey_id")["code_ligne"]
    prev_code_ligne = journey_code_ligne.shift(1)
    next_code_ligne = journey_code_ligne.shift(-1)

    depart_eligible = (data_chuuchuu["line_match_status"] == "ambiguous") & (data_chuuchuu["depart_terminus"] == "depart") & next_code_ligne.notna()
    terminus_eligible = (data_chuuchuu["line_match_status"] == "ambiguous") & (data_chuuchuu["depart_terminus"] == "terminus") & prev_code_ligne.notna()

    resolved_depart_mask = depart_eligible.copy()
    resolved_depart_mask.loc[depart_eligible] = [
        nl in c for nl, c in zip(next_code_ligne[depart_eligible], data_chuuchuu.loc[depart_eligible, "code_ligne_candidates"])
    ]
    resolved_terminus_mask = terminus_eligible.copy()
    resolved_terminus_mask.loc[terminus_eligible] = [
        pl in c for pl, c in zip(prev_code_ligne[terminus_eligible], data_chuuchuu.loc[terminus_eligible, "code_ligne_candidates"])
    ]

    data_chuuchuu.loc[resolved_depart_mask, "code_ligne"] = next_code_ligne[resolved_depart_mask]
    data_chuuchuu.loc[resolved_terminus_mask, "code_ligne"] = prev_code_ligne[resolved_terminus_mask]
    resolved_mask = resolved_depart_mask | resolved_terminus_mask
    data_chuuchuu.loc[resolved_mask, "line_match_status"] = data_chuuchuu.loc[resolved_mask, "line_match_status"] + "_resolved_by_endpoint_candidate"
    return int(resolved_mask.sum())

def resolve_chain_consistency_pass():
    """5c: intermediate stop with one resolved + one ambiguous neighbour, 3-way candidate agreement."""
    journey_code_ligne = data_chuuchuu.groupby("journey_id")["code_ligne"]
    journey_status = data_chuuchuu.groupby("journey_id")["line_match_status"]
    journey_candidates = data_chuuchuu.groupby("journey_id")["code_ligne_candidates"]

    prev_code_ligne = journey_code_ligne.shift(1)
    next_code_ligne = journey_code_ligne.shift(-1)
    prev_status = journey_status.shift(1)
    next_status = journey_status.shift(-1)
    prev_candidates = journey_candidates.shift(1)
    next_candidates = journey_candidates.shift(-1)

    base_eligible = (data_chuuchuu["line_match_status"] == "ambiguous") & (data_chuuchuu["depart_terminus"] == "intermediate")
    forward_eligible = base_eligible & next_code_ligne.notna() & (prev_status == "ambiguous")
    backward_eligible = base_eligible & prev_code_ligne.notna() & (next_status == "ambiguous")

    def check_forward(i):
        anchor = next_code_ligne.iloc[i]
        other_candidates = prev_candidates.iloc[i]
        return anchor in data_chuuchuu["code_ligne_candidates"].iloc[i] and isinstance(other_candidates, list) and anchor in other_candidates

    def check_backward(i):
        anchor = prev_code_ligne.iloc[i]
        other_candidates = next_candidates.iloc[i]
        return anchor in data_chuuchuu["code_ligne_candidates"].iloc[i] and isinstance(other_candidates, list) and anchor in other_candidates

    resolved_forward_mask = forward_eligible.copy()
    resolved_forward_mask.loc[forward_eligible] = [check_forward(i) for i in data_chuuchuu.index[forward_eligible]]
    resolved_backward_mask = backward_eligible.copy()
    resolved_backward_mask.loc[backward_eligible] = [check_backward(i) for i in data_chuuchuu.index[backward_eligible]]

    data_chuuchuu.loc[resolved_forward_mask, "code_ligne"] = next_code_ligne[resolved_forward_mask]
    data_chuuchuu.loc[resolved_backward_mask, "code_ligne"] = prev_code_ligne[resolved_backward_mask]
    resolved_mask = resolved_forward_mask | resolved_backward_mask
    data_chuuchuu.loc[resolved_mask, "line_match_status"] = data_chuuchuu.loc[resolved_mask, "line_match_status"] + "_resolved_by_chain_consistency"
    return int(resolved_mask.sum())

for iteration in range(1, MAX_CHEAP_ITERATIONS + 1):
    n_neighbours = resolve_neighbours_pass()
    n_endpoint = resolve_endpoint_pass()
    n_chain = resolve_chain_consistency_pass()
    total = n_neighbours + n_endpoint + n_chain
    print(f"pass {iteration}: neighbours={n_neighbours}, endpoint={n_endpoint}, chain={n_chain}, total={total}")
    if total == 0:
        print("no new resolutions -- converged")
        break

data_chuuchuu["line_match_status"].value_counts(dropna=False)


pass 1: neighbours=9, endpoint=17961, chain=3661, total=21631
pass 2: neighbours=0, endpoint=38, chain=0, total=38
pass 3: neighbours=0, endpoint=0, chain=0, total=0
no new resolutions -- converged


line_match_status
unique                                      5184359
ambiguous                                    563921
international                                520363
ambiguous_resolved_by_endpoint_candidate     239680
no_coordinates                               194464
ambiguous_resolved_by_neighbours             139462
no_match                                     119593
ambiguous_resolved_by_chain_consistency       70458
no_match_resolved_by_neighbours                5909
Name: count, dtype: int64

### 5e -- Resolve remaining `ambiguous` stations via topological continuity

For stations still `ambiguous` after 5a-5d, take whichever neighbouring stop (previous,
else next, on the same journey) already has a known `code_ligne` as an *anchor* line. A
candidate line is accepted if its track geometry connects to the anchor line's geometry
near the station. If exactly one candidate connects, assign it; ties or no connection
are left unresolved.

"Connects" is checked in two passes, tight first:

- **Tier 1 -- tight** (`TIGHT_SEARCH_RADIUS_M` = 500 m, exact touch): correct for the
  common case, a junction right at the station itself.
- **Tier 2 -- wide** (`WIDE_SEARCH_RADIUS_M` = 8,000 m, gap up to `WIDE_CONNECT_TOLERANCE_M`
  = 100 m), applied only to combos tier 1 couldn't resolve: catches junctions further
  out, e.g. a TGV line joining the classical network several km before a terminus --
  confirmed against an external map for Marseille Saint-Charles, where line `752000`
  (the LGV from Aix) only meets line `830000` about 6 km before the station.

Widening the search radius for *everyone* (not just tier-1 leftovers) was tried and
made things worse: a bigger zone catches more unrelated nearby lines too, turning many
previously-unique matches into ties. Cascading keeps tier 1's precision where the
junction is close, and only pays tier 2's higher tie risk where tier 1 found nothing.

**Iterated to a fixed point.** Resolving one station can turn it into a clean anchor for
its *neighbour* in the next pass -- e.g. a run of consecutive ambiguous junction
stations, where only the first one initially has a resolved stop next to it. So this
whole check re-runs, recomputing anchors from the latest `code_ligne` each time, until a
pass resolves nothing new (capped at `MAX_ITERATIONS` as a safety net; converges in 2-3
passes in practice). One caveat: tier 1 is essentially exact, but tier 2 is a looser
heuristic -- a wrong tier-2 call could in principle seed a bad anchor for the next
station in a chain, though the fast drop-off in newly-resolved rows each pass suggests
this isn't compounding badly in practice.

This is computed once per unique (station, anchor line) combination -- the result only
depends on those two things, not on any other row-level detail -- then broadcast back,
the same pattern as the station-level match table built in Steps 1-4.

Note this can't help `no_match` stations: those have zero candidates within
`MATCH_BUFFER_M` to begin with, so there's nothing for topology to test.

In [201]:
TIGHT_SEARCH_RADIUS_M = 500     # tier 1: junction expected right at the station
TIGHT_CONNECT_TOLERANCE_M = 0   # exact touch only

WIDE_SEARCH_RADIUS_M = 8000     # tier 2 fallback: junction further out (e.g. a TGV
                                 # line joining the classical network before a terminus)
WIDE_CONNECT_TOLERANCE_M = 100  # allow a small gap for imperfect digitisation

MAX_ITERATIONS = 10  # safety net -- converges in 2-3 passes in practice

line_geoms = rfn_lines_proj.dissolve(by="code_ligne")["geometry"]
station_points = stations_gdf.set_index("deutscheBahnStopId")["geometry"]

def connected_candidates(row, search_radius_m, connect_tolerance_m):
    point = station_points.get(row["deutscheBahnStopId"])
    anchor_geom = line_geoms.get(row["_anchor_line"])
    if point is None or anchor_geom is None:
        return []
    zone = point.buffer(search_radius_m)
    anchor_local = anchor_geom.intersection(zone)
    if anchor_local.is_empty:
        return []
    connected = []
    for candidate in row["code_ligne_candidates"]:
        if candidate not in line_geoms.index:
            continue
        # compute the clipped candidate geometry once -- checking is_empty and then
        # distance() separately used to call .intersection() twice per candidate
        candidate_local = line_geoms[candidate].intersection(zone)
        if not candidate_local.is_empty and candidate_local.distance(anchor_local) <= connect_tolerance_m:
            connected.append(candidate)
    return connected

def resolve_via_topology(row):
    tight_matches = connected_candidates(row, TIGHT_SEARCH_RADIUS_M, TIGHT_CONNECT_TOLERANCE_M)
    if len(tight_matches) == 1:
        return tight_matches[0]
    wide_matches = connected_candidates(row, WIDE_SEARCH_RADIUS_M, WIDE_CONNECT_TOLERANCE_M)
    return wide_matches[0] if len(wide_matches) == 1 else pd.NA

# iterate to a fixed point: resolving one station can turn it into a clean anchor for
# its neighbour in the next pass (e.g. a run of consecutive ambiguous junction stations)
total_resolved = 0
for iteration in range(1, MAX_ITERATIONS + 1):
    journey_code_ligne = data_chuuchuu.groupby("journey_id")["code_ligne"]
    prev_code_ligne = journey_code_ligne.shift(1)
    next_code_ligne = journey_code_ligne.shift(-1)
    data_chuuchuu["_anchor_line"] = prev_code_ligne.where(prev_code_ligne.notna(), next_code_ligne)

    # keep this as a real column (not a throwaway mask) so it survives the merge below
    # and stays aligned to the right rows -- otherwise the merge would also re-resolve
    # rows already settled earlier that happen to share the same (station, anchor) combo
    data_chuuchuu["_still_ambiguous"] = data_chuuchuu["line_match_status"] == "ambiguous"

    topology_inputs = (
        data_chuuchuu.loc[
            data_chuuchuu["_still_ambiguous"] & data_chuuchuu["_anchor_line"].notna(),
            ["deutscheBahnStopId", "_anchor_line", "code_ligne_candidates"],
        ]
        .drop_duplicates(subset=["deutscheBahnStopId", "_anchor_line"])
        .reset_index(drop=True)
    )
    if topology_inputs.empty:
        data_chuuchuu = data_chuuchuu.drop(columns=["_anchor_line", "_still_ambiguous"])
        print(f"iteration {iteration}: no more (station, anchor line) combos to test -- done")
        break

    topology_inputs["resolved_code_ligne"] = topology_inputs.apply(resolve_via_topology, axis=1)
    resolved_lookup = topology_inputs.loc[
        topology_inputs["resolved_code_ligne"].notna(), ["deutscheBahnStopId", "_anchor_line", "resolved_code_ligne"]
    ]

    data_chuuchuu = data_chuuchuu.merge(resolved_lookup, on=["deutscheBahnStopId", "_anchor_line"], how="left")

    topology_resolved_mask = data_chuuchuu["_still_ambiguous"] & data_chuuchuu["resolved_code_ligne"].notna()
    data_chuuchuu.loc[topology_resolved_mask, "code_ligne"] = data_chuuchuu.loc[topology_resolved_mask, "resolved_code_ligne"]
    data_chuuchuu.loc[topology_resolved_mask, "line_match_status"] = (
        data_chuuchuu.loc[topology_resolved_mask, "line_match_status"] + "_resolved_by_topology"
    )
    data_chuuchuu = data_chuuchuu.drop(columns=["_anchor_line", "_still_ambiguous", "resolved_code_ligne"])

    n_resolved = topology_resolved_mask.sum()
    total_resolved += n_resolved
    print(f"iteration {iteration}: {len(topology_inputs)} combos tested, {len(resolved_lookup)} resolved -> {n_resolved} rows")

    if n_resolved == 0:
        print("no new resolutions -- converged")
        break

print(f"\n{total_resolved} rows resolved via topological continuity across {iteration} iteration(s)")
data_chuuchuu["line_match_status"].value_counts(dropna=False)


iteration 1: 434 combos tested, 190 resolved -> 202163 rows
iteration 2: 267 combos tested, 12 resolved -> 3270 rows
iteration 3: 256 combos tested, 1 resolved -> 2 rows
iteration 4: 255 combos tested, 0 resolved -> 0 rows
no new resolutions -- converged

205435 rows resolved via topological continuity across 4 iteration(s)


line_match_status
unique                                      5184359
international                                520363
ambiguous                                    358486
ambiguous_resolved_by_endpoint_candidate     239680
ambiguous_resolved_by_topology               205435
no_coordinates                               194464
ambiguous_resolved_by_neighbours             139462
no_match                                     119593
ambiguous_resolved_by_chain_consistency       70458
no_match_resolved_by_neighbours                5909
Name: count, dtype: int64

### 5f -- Extend 5a/5b's neighbour logic to `no_match` and `no_coordinates` stops

5a-5e only ever resolve `ambiguous` stops (5a also covers `no_match`, but only its
"both neighbours agree" rule). `no_match` (coordinates exist but no line matched within
`MATCH_BUFFER_M`) and `no_coordinates` (no lat/lon at all) stops have **no candidate
list at all** -- there's no geometry to check topologically (5e) or directionally
(5g, next) either. But the same journey-continuity argument from 5a/5b still applies:
the surrounding journey tells us what line was in use, even with zero geometry for this
particular stop.

- **Intermediate stops** (5a-style): if the previous and next stop on the same journey
  are both resolved to the same `code_ligne`, assign that line here too. `no_coordinates`
  is genuinely new coverage (5a/5d never touched it); `no_match` was already looped to a
  fixed point by 5d, but running it again here still finds a few thousand more cases --
  5e's topology step runs in between and resolves additional `ambiguous` neighbours,
  which can turn a previously-unresolvable no_match stop's neighbours into an agreeing
  pair for the first time.
- **`depart`/`terminus` stops** (5b-style): only one neighbour exists. Unlike 5b's
  `ambiguous` case, there's no candidate list to corroborate the neighbour's line
  against -- so this is weaker, unverified evidence: the neighbour's `code_ligne` is
  assigned directly, trusting that a train doesn't change line between one stop and the
  very next/previous. This is new coverage for *both* `no_match` and `no_coordinates`
  (5b never touched either).

Re-run together, iterated to a fixed point (same pattern as 5d), since resolving one
`no_match`/`no_coordinates` stop can create a usable neighbour for the next one over --
e.g. a run of several consecutive foreign/unmatched stops with a resolved anchor only
at one end.

Deliberately scoped narrowly: this doesn't feed back into re-resolving `ambiguous`
stops via 5b/5c/5e, even though a newly-resolved `no_match`/`no_coordinates` neighbour
could in principle unlock one. Runs before 5g (direction of travel) simply because it's
cheap and targets a disjoint set of statuses -- order between the two doesn't otherwise
matter.

In [202]:
def resolve_neighbours_pass_no_match():
    """5a-style: both neighbours resolved and agree (extended to no_match/no_coordinates)."""
    journey_code_ligne = data_chuuchuu.groupby("journey_id")["code_ligne"]
    prev_code_ligne = journey_code_ligne.shift(1)
    next_code_ligne = journey_code_ligne.shift(-1)
    mask = (
        data_chuuchuu["line_match_status"].isin(["no_match", "no_coordinates"])
        & (data_chuuchuu["depart_terminus"] == "intermediate")
        & prev_code_ligne.notna() & next_code_ligne.notna()
        & (prev_code_ligne == next_code_ligne)
    )
    data_chuuchuu.loc[mask, "code_ligne"] = prev_code_ligne[mask]
    data_chuuchuu.loc[mask, "line_match_status"] = data_chuuchuu.loc[mask, "line_match_status"] + "_resolved_by_neighbours"
    return int(mask.sum())

def resolve_endpoint_pass_no_match():
    """5b-style: depart/terminus stop, trust the single neighbour directly (no candidates to corroborate against)."""
    journey_code_ligne = data_chuuchuu.groupby("journey_id")["code_ligne"]
    prev_code_ligne = journey_code_ligne.shift(1)
    next_code_ligne = journey_code_ligne.shift(-1)

    depart_mask = (
        data_chuuchuu["line_match_status"].isin(["no_match", "no_coordinates"])
        & (data_chuuchuu["depart_terminus"] == "depart")
        & next_code_ligne.notna()
    )
    terminus_mask = (
        data_chuuchuu["line_match_status"].isin(["no_match", "no_coordinates"])
        & (data_chuuchuu["depart_terminus"] == "terminus")
        & prev_code_ligne.notna()
    )

    data_chuuchuu.loc[depart_mask, "code_ligne"] = next_code_ligne[depart_mask]
    data_chuuchuu.loc[terminus_mask, "code_ligne"] = prev_code_ligne[terminus_mask]
    resolved_mask = depart_mask | terminus_mask
    data_chuuchuu.loc[resolved_mask, "line_match_status"] = data_chuuchuu.loc[resolved_mask, "line_match_status"] + "_resolved_by_endpoint_neighbour"
    return int(resolved_mask.sum())

MAX_NO_MATCH_ITERATIONS = 15  # safety net -- expect convergence in a handful of passes, as with 5d

for iteration in range(1, MAX_NO_MATCH_ITERATIONS + 1):
    n_neighbours = resolve_neighbours_pass_no_match()
    n_endpoint = resolve_endpoint_pass_no_match()
    total = n_neighbours + n_endpoint
    print(f"pass {iteration}: neighbours={n_neighbours}, endpoint={n_endpoint}, total={total}")
    if total == 0:
        print("no new resolutions -- converged")
        break

data_chuuchuu["line_match_status"].value_counts(dropna=False)

pass 1: neighbours=21330, endpoint=27750, total=49080
pass 2: neighbours=0, endpoint=0, total=0
no new resolutions -- converged


line_match_status
unique                                           5184359
international                                     520363
ambiguous                                         358486
ambiguous_resolved_by_endpoint_candidate          239680
ambiguous_resolved_by_topology                    205435
no_coordinates                                    160221
ambiguous_resolved_by_neighbours                  139462
no_match                                          104756
ambiguous_resolved_by_chain_consistency            70458
no_coordinates_resolved_by_neighbours              20350
no_coordinates_resolved_by_endpoint_neighbour      13893
no_match_resolved_by_endpoint_neighbour            13857
no_match_resolved_by_neighbours                     6889
Name: count, dtype: int64

### 5g -- Resolve remaining `ambiguous` stations via direction of travel

For stations still `ambiguous` after 5a-5e, use the direction the train is actually
travelling near the ambiguous stop as one more disambiguation signal: the bearing
between this station and its nearest *real* neighbouring stop on the same journey --
the previous stop if available, else the next one (for a `depart` row, which has no
previous stop). The previous stop is preferred over the next when both exist: the
direction a train just arrived from is a strong prior for the direction it continues
in, whereas the next real stop can be far enough away that the straight-line chord to
it cuts across a genuine bend in the route (see below for why that distinction
matters). A candidate line is accepted if its own track geometry, locally near the
ambiguous station, runs roughly parallel to that neighbour bearing rather than across
it.

**This corrects a real miss from an earlier version of this step**, caught by manual
inspection: that version used each journey's overall `depart` -> `terminus` straight
line as the reference axis instead of a local neighbour. On journey `TGV INOUI 6823`
(Lyon Part-Dieu -> Toulouse Matabiau), at the ambiguous stop Valence TGV
(candidates `[752000, 908000]`), the whole-journey axis (Lyon -> Toulouse, bearing
41.6 deg) favoured line `908000` ("Ligne de Valence a Moirans", tangent 54.9 deg, off
by 13.3 deg) over `752000` (the LGV continuing south, tangent 97.8 deg, off by 56.2
deg -- over the acceptance threshold, so rejected outright). But `908000` actually
heads back north-east towards Grenoble -- the wrong direction entirely -- while
`752000` is the line the train was already on. The tell: comparing against the
*previous real hop* (Lyon Part-Dieu -> Valence TGV) instead of the aggregate axis,
`752000`'s tangent is off by only 0.1 deg versus `908000`'s 42.7 deg -- a decisive,
correct call. The whole-journey axis is only a good proxy for "local direction of
travel" when the route is roughly straight; this one bends (south out of Lyon, then
west past Nimes toward Toulouse), and the coarse aggregate bearing didn't reflect the
true local direction before the bend.

Because a track is a curve, not a straight line, "the direction it's going" is only
defined *locally*: for each candidate, the nearest point on its geometry to the station
is found, then a short window (`TANGENT_WINDOW_M`) on either side of that point gives a
local tangent vector. And because a line has no inherent sign (it doesn't "know" which
way is "forward"), only the *axis* (mod 180 degrees) is comparable to the neighbour
bearing -- a candidate running exactly perpendicular to it is implausible; one running
exactly parallel (in either sign) is consistent with it.

A candidate is accepted only if it's the single best-aligned one, within
`MAX_ANGLE_DIFF_DEG` of the reference axis, with at least `MIN_MARGIN_DEG` of daylight
over the runner-up -- ties or all-candidates-off-axis are left unresolved rather than
guessed. This is still a weaker, independent signal next to 5e's exact topological
connectivity check (two lines can run parallel to each other, or to the reference
bearing, without being the same line), so it's applied last, only to what's still
`ambiguous` after everything else -- treat it as a heuristic tie-breaker, worth
spot-checking a sample of its resolutions before trusting it as much as the earlier
steps. It's also still only a single-neighbour check (unlike 5a's two-sided
agreement), so a genuinely misleading neighbour bearing (e.g. a very short hop, or a
route that bends within a few hundred metres of the station) can in principle still
fool it -- `MIN_HOP_LENGTH_M` filters out the shortest, noisiest hops, but doesn't
eliminate this risk entirely.

Stops whose nearest usable real neighbour is missing coordinates, or sits closer than
`MIN_HOP_LENGTH_M` (too close for a meaningful bearing), are skipped -- left
`ambiguous`.

In [203]:
import math

TANGENT_WINDOW_M = 500       # half-window used to estimate a candidate line's local direction
MIN_HOP_LENGTH_M = 1000      # skip a neighbour hop this short -- too close for a meaningful bearing
MAX_ANGLE_DIFF_DEG = 40      # candidate's local direction must be within this many degrees of the reference axis
MIN_MARGIN_DEG = 15          # ...and beat the runner-up candidate by at least this much

def axis_angle_deg(dx, dy):
    # Angle of a vector, folded into [0, 180) -- a line's local direction has no sign.
    return math.degrees(math.atan2(dy, dx)) % 180

def angular_diff_deg(a, b):
    diff = abs(a - b) % 180
    return min(diff, 180 - diff)

def nearest_subline(point, geom):
    # geom can be a LineString or a MultiLineString (post-dissolve) -- use whichever part is closest.
    if geom.geom_type == "LineString":
        return geom
    return min(geom.geoms, key=lambda part: part.distance(point))

def candidate_tangent_angle(stop_id, candidate):
    point = station_points.get(stop_id)
    geom = line_geoms.get(candidate)
    if point is None or geom is None:
        return None
    subline = nearest_subline(point, geom)
    if subline.length < 1:
        return None
    dist = subline.project(point)
    before = subline.interpolate(max(dist - TANGENT_WINDOW_M, 0))
    after = subline.interpolate(min(dist + TANGENT_WINDOW_M, subline.length))
    if before.distance(after) < 1:
        return None
    return axis_angle_deg(after.x - before.x, after.y - before.y)

# --- reference axis: bearing between this station and its nearest REAL neighbouring
# stop on the same journey -- previous stop preferred (arrival direction is a strong
# prior for where the train continues), next stop only as a fallback (for `depart`
# rows, which have no previous stop). Uses raw station coordinates, not code_ligne, so
# -- like 5e's anchor -- it's available even when the neighbour's own line hasn't been
# resolved. See the markdown above for why this replaced an earlier whole-journey-axis
# version that got a real case wrong.
journey_stop_ids = data_chuuchuu.groupby("journey_id")["deutscheBahnStopId"]
prev_stop_point = journey_stop_ids.shift(1).map(station_points)
next_stop_point = journey_stop_ids.shift(-1).map(station_points)
this_stop_point = data_chuuchuu["deutscheBahnStopId"].map(station_points)

def reference_axis(this_point, prev_point, next_point):
    if this_point is None:
        return None
    if prev_point is not None and this_point.distance(prev_point) >= MIN_HOP_LENGTH_M:
        return axis_angle_deg(this_point.x - prev_point.x, this_point.y - prev_point.y)
    if next_point is not None and next_point.distance(this_point) >= MIN_HOP_LENGTH_M:
        return axis_angle_deg(next_point.x - this_point.x, next_point.y - this_point.y)
    return None

still_ambiguous_mask = data_chuuchuu["line_match_status"] == "ambiguous"
reference_axis_by_row = pd.Series(
    [
        reference_axis(this_point, prev_point, next_point)
        for this_point, prev_point, next_point in zip(
            this_stop_point[still_ambiguous_mask], prev_stop_point[still_ambiguous_mask], next_stop_point[still_ambiguous_mask]
        )
    ],
    index=data_chuuchuu.index[still_ambiguous_mask],
)
print(f"{reference_axis_by_row.notna().sum()} / {still_ambiguous_mask.sum()} still-ambiguous rows have a usable neighbour-based reference axis")

# --- candidate tangent bearing: computed once per unique (station, candidate) combo --
# purely geometric, doesn't depend on the journey, so this stays cheap regardless of how
# many rows/journeys pass through a given ambiguous station
station_candidates = data_chuuchuu.loc[still_ambiguous_mask, ["deutscheBahnStopId", "code_ligne_candidates"]].drop_duplicates(
    subset=["deutscheBahnStopId"]
)

candidate_angles = station_candidates.explode("code_ligne_candidates").rename(columns={"code_ligne_candidates": "candidate"})
candidate_angles["candidate_angle_deg"] = [
    candidate_tangent_angle(stop_id, candidate)
    for stop_id, candidate in zip(candidate_angles["deutscheBahnStopId"], candidate_angles["candidate"])
]
candidate_angles = candidate_angles.dropna(subset=["candidate_angle_deg"])

candidate_angles_by_station = (
    candidate_angles.groupby("deutscheBahnStopId")
    .apply(lambda g: list(zip(g["candidate"], g["candidate_angle_deg"])))
    .to_dict()
)
print(f"local direction computed for {len(candidate_angles)} (station, candidate) pairs across {len(candidate_angles_by_station)} stations")

# --- pick the single best-aligned candidate per row ---
def best_candidate_by_direction(candidate_angle_pairs, axis_angle):
    diffs = sorted((angular_diff_deg(angle, axis_angle), candidate) for candidate, angle in candidate_angle_pairs)
    best_diff, best_candidate = diffs[0]
    if best_diff > MAX_ANGLE_DIFF_DEG:
        return pd.NA
    if len(diffs) > 1 and (diffs[1][0] - best_diff) < MIN_MARGIN_DEG:
        return pd.NA
    return best_candidate

rows_with_axis = reference_axis_by_row.dropna().index
stop_ids_with_axis = data_chuuchuu.loc[rows_with_axis, "deutscheBahnStopId"]

resolved_code_ligne = pd.Series(
    [
        best_candidate_by_direction(candidate_angles_by_station[stop_id], axis_angle)
        if stop_id in candidate_angles_by_station
        else pd.NA
        for stop_id, axis_angle in zip(stop_ids_with_axis, reference_axis_by_row.loc[rows_with_axis])
    ],
    index=rows_with_axis,
)

data_chuuchuu["_resolved_code_ligne_direction"] = pd.NA
data_chuuchuu.loc[rows_with_axis, "_resolved_code_ligne_direction"] = resolved_code_ligne

direction_resolved_mask = still_ambiguous_mask & data_chuuchuu["_resolved_code_ligne_direction"].notna()
data_chuuchuu.loc[direction_resolved_mask, "code_ligne"] = data_chuuchuu.loc[direction_resolved_mask, "_resolved_code_ligne_direction"]
data_chuuchuu.loc[direction_resolved_mask, "line_match_status"] = (
    data_chuuchuu.loc[direction_resolved_mask, "line_match_status"] + "_resolved_by_direction"
)
data_chuuchuu = data_chuuchuu.drop(columns=["_resolved_code_ligne_direction"])

print(f"{direction_resolved_mask.sum()} rows resolved via direction of travel")
data_chuuchuu["line_match_status"].value_counts(dropna=False)

328119 / 358486 still-ambiguous rows have a usable neighbour-based reference axis
local direction computed for 234 (station, candidate) pairs across 108 stations
42242 rows resolved via direction of travel


line_match_status
unique                                           5184359
international                                     520363
ambiguous                                         316244
ambiguous_resolved_by_endpoint_candidate          239680
ambiguous_resolved_by_topology                    205435
no_coordinates                                    160221
ambiguous_resolved_by_neighbours                  139462
no_match                                          104756
ambiguous_resolved_by_chain_consistency            70458
ambiguous_resolved_by_direction                    42242
no_coordinates_resolved_by_neighbours              20350
no_coordinates_resolved_by_endpoint_neighbour      13893
no_match_resolved_by_endpoint_neighbour            13857
no_match_resolved_by_neighbours                     6889
Name: count, dtype: int64

#### Spot-check a direction-resolved journey

As flagged above, this rule is a heuristic, not a geometric certainty like 5e -- worth
eyeballing a sample against the rest of the (already-resolved) itinerary before trusting
it.

In [214]:
import random
from IPython.display import display

direction_resolved_journey_ids = data_chuuchuu.loc[
    data_chuuchuu["line_match_status"] == "ambiguous_resolved_by_direction", "journey_id"
].unique()

if len(direction_resolved_journey_ids) > 0:
    sample_journey_id = random.choice(direction_resolved_journey_ids)
    sample_journey = data_chuuchuu.loc[data_chuuchuu["journey_id"] == sample_journey_id].sort_values("sort_time")
    print(f"journey_id: {sample_journey_id}  ({len(sample_journey)} stops)")
    display(
        sample_journey[
            [
                "stopName",
                "deutscheBahnStopId",
                "originalRoute",
                "depart_terminus",
                "sort_time",
                "code_ligne",
                "code_ligne_candidates",
                "line_match_status",
            ]
        ]
    )
else:
    print("no rows were resolved via direction of travel")


journey_id: FR_TRAIN TER_17353_2025-12-24  (15 stops)


,stopName,deutscheBahnStopId,originalRoute,depart_terminus,sort_time,code_ligne,code_ligne_candidates,line_match_status
5479725,Romans - Bourg-de-Péage,8700414,Train TER 17353,depart,2025-12-24 08:47:00+00:00,908000,[908000],unique
5479726,Valence TGV Rhône-Alpes Sud,8704943,Train TER 17353,intermediate,2025-12-24 08:53:00+00:00,908000,"[752000, 908000]",ambiguous_resolved_by_direction
5479727,Valence Ville,8700056,Train TER 17353,intermediate,2025-12-24 09:02:00+00:00,830000,[830000],unique
5479728,Livron,8701404,Train TER 17353,intermediate,2025-12-24 09:15:00+00:00,830000,"[830000, 830341]",ambiguous_resolved_by_topology
5479729,Crest,8702521,Train TER 17353,intermediate,2025-12-24 09:34:00+00:00,912000,[912000],unique
5479730,Saillans,8702577,Train TER 17353,intermediate,2025-12-24 09:48:00+00:00,912000,[912000],unique
5479731,Die,8702522,Train TER 17353,intermediate,2025-12-24 10:11:00+00:00,912000,[912000],unique
5479732,Luc-en-Diois,8702527,Train TER 17353,intermediate,2025-12-24 10:30:00+00:00,912000,[912000],unique
5479733,Veynes Dévoluy,8700110,Train TER 17353,intermediate,2025-12-24 11:09:00+00:00,905000,"[905000, 915000]",ambiguous_resolved_by_topology
5479734,Gap,8700111,Train TER 17353,intermediate,2025-12-24 12:12:00+00:00,915000,[915000],unique


### Verification of trains with unmatched stations:

In [205]:
import random

# Pick one journey (train) that stops at at least one ambiguous station, then look at
# its full itinerary -- the *other*, unambiguous stops on the same trip are usually
# enough to tell by eye which of the candidate lines the train was actually on.
ambiguous_journey_ids = data_chuuchuu.loc[
    data_chuuchuu["line_match_status"] == "ambiguous", "journey_id"
].unique()

sample_journey_id = random.choice(ambiguous_journey_ids)
sample_journey = data_chuuchuu.loc[data_chuuchuu["journey_id"] == sample_journey_id].sort_values("sort_time")

print(f"journey_id: {sample_journey_id}  ({len(sample_journey)} stops)")
sample_journey[
    [
        "stopName",
        "deutscheBahnStopId",
        "originalRoute",
        "depart_terminus",
        "sort_time",
        "code_ligne",
        "code_ligne_candidates",
        "line_match_status",
    ]
]


journey_id: FR_TER_832917_2025-10-06  (3 stops)


,stopName,deutscheBahnStopId,originalRoute,depart_terminus,sort_time,code_ligne,code_ligne_candidates,line_match_status
1877012,Haguenau,8700347,TER 832917,depart,2025-10-06 14:01:00+00:00,146000,"[146000, 159000]",ambiguous_resolved_by_endpoint_candidate
1877013,Bischwiller,8700428,TER 832917,intermediate,2025-10-06 14:07:00+00:00,146000,[146000],unique
1877014,Strasbourg,8700023,TER 832917,terminus,2025-10-06 14:25:00+00:00,NaN,"[070000, 142000, 145000]",ambiguous


In [206]:
no_match = data_chuuchuu[data_chuuchuu["line_match_status"] == "no_match"]
print("No match:")
print(len(no_match))
print(no_match["journey_type"].value_counts())
print("#########")

no_coordinates = data_chuuchuu[data_chuuchuu["line_match_status"] == "no_coordinates"]
print("No coordinates:")
print(len(no_coordinates))
print(no_coordinates["journey_type"].value_counts())
print("#########")

No match:
104756
journey_type
domestic         104259
international       497
Name: count, dtype: int64
#########
No coordinates:
160221
journey_type
domestic         152427
international      7794
Name: count, dtype: int64
#########


### Stamp `code_ligne` for `international` rows

Cosmetic only, applied last (after 5a-5g) so it can never leak into their
neighbour-propagation logic -- see Step 4.5's design note above.

In [207]:
international_final_mask = data_chuuchuu["line_match_status"] == "international"
data_chuuchuu.loc[international_final_mask, "code_ligne"] = "international line"

print(f"{international_final_mask.sum()} rows stamped 'international line' in code_ligne")
data_chuuchuu["line_match_status"].value_counts(dropna=False)

520363 rows stamped 'international line' in code_ligne


line_match_status
unique                                           5184359
international                                     520363
ambiguous                                         316244
ambiguous_resolved_by_endpoint_candidate          239680
ambiguous_resolved_by_topology                    205435
no_coordinates                                    160221
ambiguous_resolved_by_neighbours                  139462
no_match                                          104756
ambiguous_resolved_by_chain_consistency            70458
ambiguous_resolved_by_direction                    42242
no_coordinates_resolved_by_neighbours              20350
no_coordinates_resolved_by_endpoint_neighbour      13893
no_match_resolved_by_endpoint_neighbour            13857
no_match_resolved_by_neighbours                     6889
Name: count, dtype: int64

## Exporting data

In [208]:
export_data = input("Export intermediate data to parquet? (y/n): ")

if export_data.lower() == "y":
    os.makedirs(intermediate_outputs_dir, exist_ok=True)

    # code_ligne_candidates is a list column -- parquet handles it fine via pyarrow,
    # but keep it in mind if this file is later read with an engine that doesn't
    data_chuuchuu.to_parquet(f"{intermediate_outputs_dir}/data_chuuchuu_{data_selection}_lines.parquet")
